# Elhub 2021 → Cassandra → Spark → Plots → MongoDB

Denne notebooken gjør hele oppgaven:

1. **Hent** timevis produksjonsdata for *alle* prisområder fra Elhub `PRODUCTION_PER_GROUP_MBA_HOUR` for **hele 2021**, måned for måned (Usage Guidelines: én måned per kall, **inkluderende** start og slutt, fast offset `+00:00` for å unngå DST-feil).
2. **Ekstrahér** kun listen `productionPerGroupMbaHour`, normaliser til `priceArea`, `productionGroup`, `startTime`, `quantityKwh` og lag en **Spark DataFrame**.
3. **Skriv** til Cassandra-tabellen `elhub_data.production_hourly_by_group` (PK ((pricearea, productiongroup), starttime)).
4. **Les** samme 4 kolonner fra Cassandra.
5. **Plott**:
   - Kake: total produksjon for året for et valgt prisområde (ett kakestykke pr. gruppe).
   - Linje: første måned for valgt prisområde, én linje per gruppe.
6. **Lagre** Spark-data til **MongoDB Atlas**.


In [1]:
# === Parametre ===
YEAR = 2021
CHOSEN_AREA = "NO1"
FIRST_MONTH = 1

# Cassandra
CASSANDRA_HOST = "127.0.0.1"
CASSANDRA_PORT = "9042"
CASSANDRA_KEYSPACE = "elhub_data"
CASSANDRA_TABLE = "production_hourly_by_group"

# MongoDB (Atlas)
MONGO_URI = (
    "mongodb+srv://AHS_db_user:"
    "fAdp5LC8GglDRedl"
    "@ahs786student.qh8rsrb.mongodb.net/elhub"
    "?retryWrites=true&w=majority&tls=true&appName=AHS786Student"
)
MONGO_DB = "elhub"
MONGO_COLL = "production_2021_hourly_by_group"

# Elhub API
BASE_URL = "https://api.elhub.no/energy-data/v0/price-areas"
DATASET  = "PRODUCTION_PER_GROUP_MBA_HOUR"


In [2]:
# === SparkSession m/ Cassandra- og Mongo-connectorer ===
from pyspark.sql import SparkSession
cassandra_pkg = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"
mongo_pkg     = "org.mongodb.spark:mongo-spark-connector_2.12:10.5.0"

spark = (
    SparkSession.builder
    .appName("ELHUB_2021_to_Cassandra_Mongo_Plots")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.jars.packages", f"{cassandra_pkg},{mongo_pkg}")
    .config("spark.cassandra.connection.host", CASSANDRA_HOST)
    .config("spark.cassandra.connection.port", CASSANDRA_PORT)
    .config("spark.driver.extraJavaOptions", "-Djdk.tls.client.protocols=TLSv1.2")
    .config("spark.executor.extraJavaOptions", "-Djdk.tls.client.protocols=TLSv1.2")
    .getOrCreate()
)
spark


:: loading settings :: url = jar:file:/Users/a.h.sheikh/.pyenv/versions/3.12.6/envs/ind320/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/a.h.sheikh/.ivy2/cache
The jars for the packages stored in: /Users/a.h.sheikh/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5b2e5b5f-543b-4f30-97cb-201213908bf7;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#H

In [3]:
# === Hjelpefunksjoner ===
import requests, time, calendar
from typing import Dict, List, Tuple
from datetime import datetime, timezone
from pyspark.sql import functions as F, types as T

def month_utc_range(year: int, month: int) -> Tuple[str, str]:
    start_local = datetime(year, month, 1, 0, 0, 0)
    last_day = calendar.monthrange(year, month)[1]
    end_local = datetime(year, month, last_day, 23, 0, 0)
    start_utc = start_local.replace(tzinfo=timezone.utc)
    end_utc   = end_local.replace(tzinfo=timezone.utc)
    fmt = "%Y-%m-%dT%H:%M:%S+00:00"
    return start_utc.strftime(fmt), end_utc.strftime(fmt)

def _merge_with_area(items: List[Dict], pa_code: str) -> List[Dict]:
    out: List[Dict] = []
    for it in items:
        if isinstance(it, dict):
            if "priceArea" not in it and pa_code:
                it = dict(it); it["priceArea"] = pa_code
            out.append(it)
    return out

def _extract_prod_list(js: Dict) -> List[Dict]:
    items = js.get("productionPerGroupMbaHour")
    if isinstance(items, list): return items
    data = js.get("data")

    if isinstance(data, dict):
        items = data.get("productionPerGroupMbaHour")
        if isinstance(items, list): return items
        pa_list = data.get("priceAreas")
        if isinstance(pa_list, list):
            merged: List[Dict] = []
            for pa in pa_list:
                if not isinstance(pa, dict): continue
                pa_code = pa.get("priceArea") or pa.get("code") or pa.get("name")
                li = pa.get("productionPerGroupMbaHour") or (pa.get("attributes", {}) if isinstance(pa.get("attributes", {}), dict) else {}).get("productionPerGroupMbaHour")
                if isinstance(li, list): merged.extend(_merge_with_area(li, pa_code))
            if merged: return merged

    if isinstance(data, list):
        merged: List[Dict] = []
        for obj in data:
            if not isinstance(obj, dict): continue
            attrs = obj.get("attributes", {}) if isinstance(obj.get("attributes", {}), dict) else {}
            pa_code = obj.get("priceArea") or attrs.get("priceArea")
            li = obj.get("productionPerGroupMbaHour") or attrs.get("productionPerGroupMbaHour")
            if isinstance(li, list):
                merged.extend(_merge_with_area(li, pa_code)); continue
            pa_list = attrs.get("priceAreas")
            if isinstance(pa_list, list):
                for pa in pa_list:
                    if not isinstance(pa, dict): continue
                    pa_code2 = pa.get("priceArea") or pa.get("code") or pa.get("name")
                    li2 = pa.get("productionPerGroupMbaHour") or (pa.get("attributes", {}) if isinstance(pa.get("attributes", {}), dict) else {}).get("productionPerGroupMbaHour")
                    if isinstance(li2, list): merged.extend(_merge_with_area(li2, pa_code2))
        if merged: return merged

    top_keys = list(js.keys())
    dtype = type(data).__name__
    sample = None
    if isinstance(data, list) and data: sample = list(data[0].keys())
    raise RuntimeError(f"Fant ikke 'productionPerGroupMbaHour'. Toppnøkler={top_keys}, data-type={dtype}, sample0-keys={sample}")

def fetch_all_pages(url: str, params: Dict) -> List[Dict]:
    rows: List[Dict] = []
    s = requests.Session(); s.headers.update({"Accept":"application/json"})
    next_url, next_params = url, params.copy()
    while True:
        r = s.get(next_url, params=next_params, timeout=60)
        r.raise_for_status()
        js = r.json()
        items = _extract_prod_list(js)
        if not isinstance(items, list): raise RuntimeError("productionPerGroupMbaHour ble ikke en liste etter ekstraksjon.")
        rows.extend(items)
        links = js.get("links") or {}; nxt = links.get("next")
        if not nxt: break
        next_url, next_params = nxt, {}; time.sleep(0.05)
    return rows

def normalize_rows(items: List[Dict]) -> List[Dict]:
    out: List[Dict] = []
    for it in items:
        if not isinstance(it, dict): continue
        price_area = it.get("priceArea") or it.get("price_area")
        prod_group = it.get("productionGroup") or it.get("production_group")
        start_time = it.get("startTime") or it.get("start_time") or it.get("start")
        qty        = it.get("quantityKwh") or it.get("quantitykwh") or it.get("quantity")
        if None in (price_area, prod_group, start_time, qty): continue
        out.append({"priceArea":price_area,"productionGroup":prod_group,"startTime":start_time,"quantityKwh":qty})
    return out


In [4]:
# === Hent 2021, skriv til Cassandra ===
from pyspark.sql import functions as F, types as T
total_written = 0
for m in range(1, 13):
    start_iso, end_iso = month_utc_range(YEAR, m)
    params = {"dataset": DATASET, "startDate": start_iso, "endDate": end_iso}
    print(f"Henter {YEAR}-{m:02d}  [{start_iso} → {end_iso}] ...", flush=True)
    items = fetch_all_pages(BASE_URL, params)
    rows  = normalize_rows(items)
    sdf = (
        spark.createDataFrame(rows)
             .select("priceArea","productionGroup","startTime","quantityKwh")
             .withColumn("starttime", F.to_timestamp("startTime"))
             .withColumn("pricearea", F.col("priceArea"))
             .withColumn("productiongroup", F.col("productionGroup"))
             .withColumn("quantitykwh", F.col("quantityKwh").cast(T.DoubleType()))
             .select("pricearea","productiongroup","starttime","quantitykwh")
    )
    (sdf.write.format("org.apache.spark.sql.cassandra")
        .mode("append")
        .options(keyspace=CASSANDRA_KEYSPACE, table=CASSANDRA_TABLE)
        .save())
    n = sdf.count(); total_written += n
    print(f"  Rader denne måneden (etter rens): {n:,}")
print(f"Ferdig. Totalt skrevet: {total_written:,} rader → {CASSANDRA_KEYSPACE}.{CASSANDRA_TABLE}")


Henter 2021-01  [2021-01-01T00:00:00+00:00 → 2021-01-31T23:00:00+00:00] ...


  Rader denne måneden (etter rens): 13,876
Henter 2021-02  [2021-02-01T00:00:00+00:00 → 2021-02-28T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 13,106
Henter 2021-03  [2021-03-01T00:00:00+00:00 → 2021-03-31T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 15,072
Henter 2021-04  [2021-04-01T00:00:00+00:00 → 2021-04-30T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 15,548
Henter 2021-05  [2021-05-01T00:00:00+00:00 → 2021-05-31T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 16,103
Henter 2021-06  [2021-06-01T00:00:00+00:00 → 2021-06-30T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 15,701
Henter 2021-07  [2021-07-01T00:00:00+00:00 → 2021-07-31T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 16,176
Henter 2021-08  [2021-08-01T00:00:00+00:00 → 2021-08-31T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 16,073
Henter 2021-09  [2021-09-01T00:00:00+00:00 → 2021-09-30T23:00:00+00:00] ...
  Rader denne måneden (etter rens): 15,342
Hente

In [5]:
# === Les 4 kolonner fra Cassandra ===
from pyspark.sql import functions as F
cdf = (
    spark.read.format("org.apache.spark.sql.cassandra")
    .options(keyspace=CASSANDRA_KEYSPACE, table=CASSANDRA_TABLE)
    .load()
    .select("pricearea","productiongroup","starttime","quantitykwh")
    .withColumnRenamed("pricearea","priceArea")
    .withColumnRenamed("productiongroup","productionGroup")
    .withColumnRenamed("starttime","startTime")
    .withColumnRenamed("quantitykwh","quantityKwh")
)
cdf.printSchema(); cdf.show(5, truncate=False)


root
 |-- priceArea: string (nullable = false)
 |-- productionGroup: string (nullable = false)
 |-- startTime: timestamp (nullable = true)
 |-- quantityKwh: double (nullable = true)

+---------+---------------+-------------------+-----------+
|priceArea|productionGroup|startTime          |quantityKwh|
+---------+---------------+-------------------+-----------+
|NO5      |other          |2021-12-31 20:00:00|0.0        |
|NO5      |other          |2021-12-31 19:00:00|0.0        |
|NO5      |other          |2021-12-31 18:00:00|0.0        |
|NO5      |other          |2021-12-31 17:00:00|0.0        |
|NO5      |other          |2021-12-31 16:00:00|0.0        |
+---------+---------------+-------------------+-----------+
only showing top 5 rows



In [6]:
# === Plots (lagres til figs/) ===
import os, calendar as _cal
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from datetime import datetime as _dt, timezone as _tz
from pyspark.sql import functions as F

os.makedirs("figs", exist_ok=True)

year_area = (
    cdf.filter(F.col("priceArea")==CHOSEN_AREA)
       .groupBy("productionGroup")
       .agg(F.sum("quantityKwh").alias("totalKwh"))
       .orderBy("productionGroup")
)
pdf_pie = year_area.toPandas()
plt.figure()
plt.pie(pdf_pie["totalKwh"], labels=pdf_pie["productionGroup"], autopct="%1.1f%%")
plt.title(f"Total produksjon {YEAR} – {CHOSEN_AREA}")
pie_path = os.path.join("figs", f"pie_total_{YEAR}_{CHOSEN_AREA}.png")
plt.savefig(pie_path, dpi=160, bbox_inches="tight"); plt.close()
print("Lagret kake-plot til:", pie_path)

month_start = _dt(YEAR, FIRST_MONTH, 1, 0, 0, 0, tzinfo=_tz.utc)
last_day = _cal.monthrange(YEAR, FIRST_MONTH)[1]
month_end   = _dt(YEAR, FIRST_MONTH, last_day, 23, 0, 0, tzinfo=_tz.utc)

jan = (
    cdf.filter((F.col("priceArea")==CHOSEN_AREA) &
               (F.col("startTime") >= F.lit(month_start)) &
               (F.col("startTime") <= F.lit(month_end)))
       .groupBy("startTime","productionGroup")
       .agg(F.sum("quantityKwh").alias("kwh"))
)
jan_wide = (jan.groupBy("startTime").pivot("productionGroup").agg(F.first("kwh")).orderBy("startTime"))
pdf_line = jan_wide.toPandas().set_index("startTime")
plt.figure(figsize=(11,5))
for col in pdf_line.columns: plt.plot(pdf_line.index, pdf_line[col], label=col)
plt.title(f"Timevis produksjon – {CHOSEN_AREA}, {month_start.strftime('%B %Y')}")
plt.xlabel("Tid (UTC)"); plt.ylabel("kWh"); plt.legend(title="Production Group"); plt.tight_layout()
line_path = os.path.join("figs", f"line_{YEAR}_{CHOSEN_AREA}_{FIRST_MONTH:02d}.png")
plt.savefig(line_path, dpi=160, bbox_inches="tight"); plt.close()
print("Lagret linje-plot til:", line_path)


Lagret kake-plot til: figs/pie_total_2021_NO1.png
Lagret linje-plot til: figs/line_2021_NO1_01.png


In [7]:
# === Skriv til MongoDB ===
from pyspark.sql import functions as F
cdf_out = (
    cdf.withColumn(
        "_id",
        F.concat_ws("#",
            F.col("priceArea"),
            F.col("productionGroup"),
            F.date_format("startTime", "yyyy-MM-dd'T'HH:mm:ssXXX")
        )
    )
)
(cdf_out.write.format("mongodb")
    .mode("overwrite")   # bruk 'append' om du ikke vil overskrive
    .option("uri", MONGO_URI)
    .option("database", MONGO_DB)
    .option("collection", MONGO_COLL)
    .save())
print(f"Skrevet til MongoDB → {MONGO_DB}.{MONGO_COLL}")


Skrevet til MongoDB → elhub.production_2021_hourly_by_group
